In [5]:
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np

client = chromadb.PersistentClient(path="E:\\P1_JOB\\AI projects\\financial_risk_rag\\chroma_db")
collection = client.get_collection("financial_risk_10k")
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# CHECK 1: Counts match
print(f"ChromaDB items: {collection.count()}")
# Should match your chunking_report total

# CHECK 2: Embedding model matches
metadata = collection.metadata
print(f"Stored embedding model: {metadata.get('embedding_model')}")
print(f"Current model: BAAI/bge-small-en-v1.5")
# Should be identical — if not, you have a model mismatch bug

# CHECK 3: Sample embedding looks reasonable
sample = collection.get(limit=1, include=['embeddings'])
emb = np.array(sample['embeddings'][0])
print(f"\nSample embedding dimension: {len(emb)}")
print(f"Norm: {np.linalg.norm(emb):.4f}")  # Should be ~1.0 if normalized
print(f"Min value: {emb.min():.4f}, Max value: {emb.max():.4f}")
print(f"Any NaN? {np.isnan(emb).any()}")
# Norm ~1.0 → normalized. NaN → bug. Min/max in roughly [-0.3, 0.3] is typical.

# CHECK 4: Zero-vector check (silent failure mode)
sample_100 = collection.get(limit=100, include=['embeddings'])
norms = [np.linalg.norm(e) for e in sample_100['embeddings']]
zero_count = sum(1 for n in norms if n < 0.01)
print(f"\nZero-vector chunks in sample: {zero_count}/100")
# Should be 0. Any > 0 indicates extraction-empty chunks got embedded.

# CHECK 5: Metadata completeness
sample_meta = collection.get(limit=10, include=['metadatas'])
print(f"\nSample metadata keys: {list(sample_meta['metadatas'][0].keys())}")
print(f"Sample metadata: {sample_meta['metadatas'][0]}")
# Should include all your expected fields

# CHECK 6: Round-trip sanity check
test_text = "credit risk and counterparty exposure"
test_emb = model.encode(test_text, normalize_embeddings=True)

# Re-encode the same text — should produce identical embedding
test_emb2 = model.encode(test_text, normalize_embeddings=True)
similarity = np.dot(test_emb, test_emb2)
print(f"\nSame text re-encoded similarity: {similarity:.6f}")
# Should be 1.0 (or extremely close). If not, model behavior is non-deterministic.

# CHECK 7: Known-good query returns reasonable results
results = collection.query(
    query_embeddings=[test_emb.tolist()],
    n_results=3
)

print(f"\nQuery: '{test_text}'")
for i in range(3):
    print(f"\n#{i+1} distance: {results['distances'][0][i]:.4f}")
    print(f"   Source: {results['metadatas'][0][i].get('source')}")
    print(f"   Section: {results['metadatas'][0][i].get('section')}")
    print(f"   {results['documents'][0][i][:200]}...")
# Distances should be < 0.7 for top-1, results should be about credit risk

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ChromaDB items: 1401
Stored embedding model: None
Current model: BAAI/bge-small-en-v1.5

Sample embedding dimension: 384
Norm: 1.0000
Min value: -0.3453, Max value: 0.3705
Any NaN? False

Zero-vector chunks in sample: 0/100

Sample metadata keys: ['ticker', 'char_start', 'filing_year', 'chunk_index', 'item', 'source', 'file_type', 'char_end', 'company']
Sample metadata: {'ticker': 'AXP', 'char_start': 0, 'filing_year': 2026, 'chunk_index': 0, 'item': '1A', 'source': 'AXP_0000004962-26-000080_risk_factors.txt', 'file_type': 'risk_factors', 'char_end': 1184, 'company': 'American Express'}

Same text re-encoded similarity: 1.000000

Query: 'credit risk and counterparty exposure'

#1 distance: 0.1467
   Source: COF_0000927628-26-000024_risk_factors.txt
   Section: None
   •
Counterparty Credit Risk
: In addition to consumer credit risk, we are exposed to counterparty credit risk arising from business-facing activities, including our operation of the Global Payment Netw...

#2 distance: 0.1